# Pré sélection de variable

Cette sélection c'est bassé sur le fichier JSON fournis par l'ADEME sur les colones des dpe existants.

Ainsi, via des groupes déjà existant, la sélection de variable à pu se faire plus facilement.

Cette sélection n'a pas vocation d'être exaustif, elle faciliterra simplement le travail suivant

## Exploration des variables

In [1]:
import json

# Récupération des informations du json
with open('dpe03existant-schema.json', 'r', encoding='utf-8') as f:
    data_json = json.load(f)

print(data_json) # On s'intéresse au "properties"

{'type': 'object', 'required': [], 'properties': {'numero_dpe': {'x-originalName': 'numero_dpe', 'type': 'string', 'description': '', 'x-group': 'Administratif', 'x-cardinality': 15957008, 'title': '', 'x-capabilities': {}, 'x-refersTo': 'Numero_DPE', 'x-concept': {'id': 'Numero_DPE', 'title': 'Numéro de DPE', 'primary': True}}, 'date_derniere_modification_dpe': {'x-originalName': 'date_derniere_modification_dpe', 'type': 'string', 'format': 'date', 'title': '', 'dateFormat': 'D/M/YYYY', 'description': "Date de dernière modification d'une donnée concernant le DPE. A utiliser pour mettre en place une alimentation incrémentale.", 'x-group': 'Administratif', 'x-cardinality': 318, 'x-capabilities': {'textStandard': False}}, 'date_visite_diagnostiqueur': {'x-originalName': 'date_visite_diagnostiqueur', 'type': 'string', 'format': 'date', 'dateFormat': 'D/M/YYYY', 'description': '', 'x-group': 'Administratif', 'x-cardinality': 1989, 'title': '', 'x-capabilities': {'textStandard': False}}, 'd

In [2]:
import pandas as pd
data = pd.DataFrame(data_json["properties"]).transpose() 
data.head()
# On a en index les différentes variables et en colone les infos sur les variables 

,x-originalName,type,description,x-group,x-cardinality,title,x-capabilities,x-refersTo,x-concept,format,dateFormat,maxLength,examples
numero_dpe,numero_dpe,string,,Administratif,15957008,,{},Numero_DPE,"{'id': 'Numero_DPE', 'title': 'Numéro de DPE',...",NaN,NaN,NaN,NaN
date_derniere_modification_dpe,date_derniere_modification_dpe,string,Date de dernière modification d'une donnée con...,Administratif,318,,{'textStandard': False},NaN,NaN,date,D/M/YYYY,NaN,NaN
date_visite_diagnostiqueur,date_visite_diagnostiqueur,string,,Administratif,1989,,{'textStandard': False},NaN,NaN,date,D/M/YYYY,NaN,NaN
date_etablissement_dpe,date_etablissement_dpe,string,,Administratif,1960,,{'textStandard': False},NaN,NaN,date,D/M/YYYY,NaN,NaN
date_reception_dpe,date_reception_dpe,string,,Administratif,1958,,{'textStandard': False},NaN,NaN,date,D/M/YYYY,NaN,NaN


In [3]:
data["x-group"].unique() # Les groupes de variables qui vont aidé à une pré sélection de variables

array(['Administratif', 'Bilan DPE', 'Caractéristiques bâtiment',
       'Localisation', "Confort d'été", 'Déperdition', 'Isolation',
       'Apport et besoin', 'Consommation en énergie primaire',
       'Consommation en énergie finale', 'Emissions de GES',
       'Bilan par énergie', 'Coûts', 'Chauffage', 'ECS', 'Ventilation',
       'Climatisation', 'Energie Renouvelable'], dtype=object)

In [4]:
# L'administratif ne sera pas à garder pour les analyses
administratif = data[data["x-group"] == "Administratif"].index.tolist()
administratif = administratif[1:] # On retirer le numéro DPE qu'on gardera en index (vu qu'il est un identifiant unique)
administratif

['date_derniere_modification_dpe',
 'date_visite_diagnostiqueur',
 'date_etablissement_dpe',
 'date_reception_dpe',
 'date_fin_validite_dpe',
 'numero_dpe_remplace',
 'numero_dpe_immeuble_associe',
 'id_rnb',
 'provenance_id_rnb',
 'numero_rpls_logement',
 'numero_immatriculation_copropriete',
 'modele_dpe',
 'version_dpe',
 'methode_application_dpe']

In [5]:
# DPE -> notre variable cible quali
dpe = data[data["x-group"] == "Bilan DPE"].index.tolist()
dpe

['etiquette_dpe', 'etiquette_ges']

In [6]:
# Consomation -> notre variable cible quanti
consomation = data[(data["x-group"] == "Consommation en énergie primaire") | (data["x-group"] == "Consommation en énergie finale")].index.tolist()
consomation 

['conso_5_usages_ep',
 'conso_5_usages_par_m2_ep',
 'conso_chauffage_ep',
 'conso_ecs_ep',
 'conso_refroidissement_ep',
 'conso_eclairage_ep',
 'conso_auxiliaires_ep',
 'conso_5_usages_ef',
 'conso_5_usages_par_m2_ef',
 'conso_chauffage_ef',
 'conso_ecs_ef',
 'conso_refroidissement_ef',
 'conso_eclairage_ef',
 'conso_auxiliaires_ef']

In [7]:
localisation = data[data["x-group"] == "Localisation"].index.tolist()
localisation 
# Les deux première variable peuvent se révélé intéressante pour la prédiction
# On va égalament garder les coordonnee_cartographique pouvant se révélé utile pour les cartes
# Le reste pourra être supprimé
variable_a_garder = ["classe_altitude","zone_climatique","coordonnee_cartographique_x_ban","coordonnee_cartographique_y_ban"]
localisation = [x for x in localisation if x not in variable_a_garder]
localisation

['adresse_ban',
 'numero_voie_ban',
 'nom_rue_ban',
 'nom_commune_ban',
 'code_postal_ban',
 'code_insee_ban',
 'code_departement_ban',
 'code_region_ban',
 'identifiant_ban',
 'score_ban',
 'statut_geocodage',
 'adresse_brut',
 'adresse_complete_brut',
 'nom_commune_brut',
 'code_postal_brut',
 'numero_etage_appartement',
 'position_logement_dans_immeuble',
 'nom_residence',
 'complement_adresse_batiment',
 'complement_adresse_logement']

In [8]:
# Information inconnue par les utilisateurs 
apport_et_besoin = data[data["x-group"] == "Apport et besoin"].index.tolist()
apport_et_besoin

['besoin_chauffage',
 'besoin_ecs',
 'besoin_refroidissement',
 'apport_interne_saison_chauffe',
 'apport_interne_saison_froide',
 'apport_solaire_saison_chauffe',
 'apport_solaire_saison_froide']

In [9]:
# Les émmissions ne sont pas des informations que les utilisateurs peuvent avoir
# Et sont trop lié à l'étiquette DPE (aide à son calcul)
emission = data[data["x-group"] == "Emissions de GES"].index.tolist()
emission

['emission_ges_5_usages',
 'emission_ges_5_usages_par_m2',
 'emission_ges_chauffage',
 'emission_ges_ecs',
 'emission_ges_refroidissement',
 'emission_ges_eclairage',
 'emission_ges_auxiliaires']

In [10]:
#Idem pour bilan énergie et cout
bilan_energie = data[data["x-group"] == "Bilan par énergie"].index.tolist()
bilan_energie

['type_energie_n1',
 'conso_5_usages_ef_energie_n1',
 'conso_chauffage_ef_energie_n1',
 'conso_ecs_ef_energie_n1',
 'cout_total_5_usages_energie_n1',
 'cout_chauffage_energie_n1',
 'cout_ecs_energie_n1',
 'emission_ges_5_usages_energie_n1',
 'emission_ges_chauffage_energie_n1',
 'emission_ges_ecs_energie_n1',
 'type_energie_n2',
 'conso_5_usages_ef_energie_n2',
 'conso_chauffage_ef_energie_n2',
 'conso_ecs_ef_energie_n2',
 'cout_total_5_usages_energie_n2',
 'cout_chauffage_energie_n2',
 'cout_ecs_energie_n2',
 'emission_ges_5_usages_energie_n2',
 'emission_ges_chauffage_energie_n2',
 'emission_ges_ecs_energie_n2',
 'type_energie_n3',
 'conso_5_usages_ef_energie_n3',
 'conso_chauffage_ef_energie_n3',
 'conso_ecs_ef_energie_n3',
 'cout_total_5_usages_energie_n3',
 'cout_chauffage_energie_n3',
 'cout_ecs_energie_n3',
 'emission_ges_5_usages_energie_n3',
 'emission_ges_chauffage_energie_n3',
 'emission_ges_ecs_energie_n3']

In [11]:
cout = data[data["x-group"] == "Coûts"].index.tolist()
cout

['cout_total_5_usages',
 'cout_chauffage',
 'cout_ecs',
 'cout_refroidissement',
 'cout_eclairage',
 'cout_auxiliaires']

In [12]:
variable_a_suppr = administratif + apport_et_besoin + cout + bilan_energie + emission + localisation
variable_cible = dpe + consomation

## Enregistrement dans des fichier txt

In [13]:
with open("variable_a_suppr.txt", "w", encoding="utf-8") as f:
    for x in variable_a_suppr:
        f.write(x + "\n")

In [14]:
with open("variable_cible.txt", "w", encoding="utf-8") as f:
    for x in variable_cible:
        f.write(x + "\n")

In [15]:
with open("variable_illu.txt", "w", encoding="utf-8") as f:
    for x in variable_a_garder:
        f.write(x + "\n")